In [0]:
%pip install -q "monai[nibabel,tqdm,einops]>=1.4" einops "nibabel>=5.2" "connected-components-3d>=3.12" "scipy>=1.11" "scikit-learn>=1.4" "matplotlib>=3.8" "pyyaml>=6.0" "openpyxl>=3.1" -e /Workspace/Users/soumyanil.banerjee@corewellhealth.org/Brain_Tumor_Segmentation

In [0]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [0]:
import subprocess, sys, os
import mlflow

# End any stale active run from a previous execution
mlflow.end_run()

os.chdir("/Workspace/Users/soumyanil.banerjee@corewellhealth.org/Brain_Tumor_Segmentation")

# --- Data & auth setup ---
os.environ["BRATS_CACHE_ROOT"] = "/Volumes/imaging_research/brain_tumor_segmentation/brats_dataset"
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
os.environ["DATABRICKS_HOST"] = ctx.apiUrl().get()
os.environ["DATABRICKS_TOKEN"] = ctx.apiToken().get()
os.environ["MLFLOW_TRACKING_URI"] = "databricks"

# Disable system-metrics monitor — its trace data causes the experiment page
# to render the GenAI observability view instead of classic Runs / Charts.
os.environ["MLFLOW_ENABLE_SYSTEM_METRICS_LOGGING"] = "false"

# --- Start MLflow run from the notebook ---
# By NOT calling mlflow.set_experiment(), the run goes into this notebook's
# built-in experiment, which shows live charts in the sidebar (flask icon).
run = mlflow.start_run(run_name="segresnet_base")
run_id = run.info.run_id
print(f"MLflow run started  run_id={run_id}")
print(f"Experiment ID: {run.info.experiment_id}")
print("Click the flask icon in the right sidebar to see live training curves.")

# Pass the run ID to the subprocess so it logs into the same run.
os.environ["MLFLOW_RUN_ID"] = run_id

# --- Launch training ---
result = subprocess.run(
    [sys.executable, "-m", "brats.train", "--config", "configs/segresnet_base.yaml", "--resume"],
    cwd="/Workspace/Users/soumyanil.banerjee@corewellhealth.org/Brain_Tumor_Segmentation",
    env=os.environ.copy(),
)
print(f"\nTraining exit code: {result.returncode}")

# --- End the MLflow run ---
mlflow.end_run()
print(f"MLflow run {run_id} finished.")

In [0]:
import mlflow
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import tempfile, os
from monai.inferers import sliding_window_inference
from brats.models.multitask import load_model_from_checkpoint
from brats.data.preprocess import dequantize, labels_to_regions
from brats.constants import REGIONS, COHORTS

# ------------------------------------------------------------------ #
# Resume the training run (started in the cell above)                #
# ------------------------------------------------------------------ #
if "run_id" not in dir():
    raise RuntimeError("Run the training cell first to set run_id.")

CACHE_ROOT = "/Volumes/imaging_research/brain_tumor_segmentation/brats_dataset"
SPLITS_CSV = "/Workspace/Users/soumyanil.banerjee@corewellhealth.org/Brain_Tumor_Segmentation/splits/split_random_seed42.csv"
CKPT      = "/Workspace/Users/soumyanil.banerjee@corewellhealth.org/Brain_Tumor_Segmentation/checkpoints/segresnet_base/best_avg.pt"

# ---- Load best model ----
model = load_model_from_checkpoint(CKPT, map_location="cuda:0").cuda().eval()
ckpt  = torch.load(CKPT, map_location="cuda:0", weights_only=False)
print(f"Loaded checkpoint from epoch {ckpt.get('epoch','?')}")

# ---- Select two largest-tumor val cases per cohort ----
df     = pd.read_csv(SPLITS_CSV)
val_df = df[df["split"] == "val"]

candidates = {c: [] for c in COHORTS}
for _, row in val_df.iterrows():
    npz_path = f"{CACHE_ROOT}/{row['cohort']}/{row['case_id']}.npz"
    with np.load(npz_path) as z:
        if "seg" in z.files:
            candidates[row["cohort"]].append({
                "case_id": row["case_id"], "cohort": row["cohort"],
                "path": npz_path, "wt": int((z["seg"] > 0).sum()),
            })

selected = []
for c in COHORTS:
    top2 = sorted(candidates[c], key=lambda x: x["wt"], reverse=True)[:2]
    selected.extend(top2)
    for i, s in enumerate(top2):
        print(f"  {c} #{i+1}: {s['case_id']} (WT={s['wt']:,})")

# ---- Run inference & collect dice scores ----
def dice_score(pred, gt):
    eps = 1e-7
    inter = (pred * gt).sum()
    return float((2 * inter + eps) / (pred.sum() + gt.sum() + eps))

def model_fn(patch):
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
        out = model(patch, gt_wt=None)
    return out["seg"]

results = []
for rec in selected:
    with np.load(rec["path"]) as z:
        img = dequantize(z["img"]).astype(np.float32)
        seg = labels_to_regions(z["seg"]).astype(np.float32)
    x    = torch.from_numpy(img).unsqueeze(0).cuda()
    pred = sliding_window_inference(x, (128, 128, 128), sw_batch_size=4,
                                    predictor=model_fn, overlap=0.5, mode="gaussian")
    pred = (torch.sigmoid(pred[0]).cpu().numpy() > 0.5).astype(np.float32)
    dice = {r: dice_score(pred[i], seg[i]) for i, r in enumerate(REGIONS)}
    results.append({"case_id": rec["case_id"], "cohort": rec["cohort"],
                    "img": img, "seg_gt": seg, "seg_pred": pred, "dice": dice})
    print(f"  {rec['cohort']}/{rec['case_id']}: "
          f"ET={dice['ET']:.3f}  TC={dice['TC']:.3f}  WT={dice['WT']:.3f}")

# ---- Visualization helpers ----
def find_tumor_center(wt):
    coords = np.argwhere(wt > 0)
    return tuple(coords.mean(0).astype(int)) if len(coords) else tuple(s // 2 for s in wt.shape)

def overlay(img_sl, seg_sl, alpha=0.55):
    lo, hi = (np.percentile(img_sl[img_sl > 0], [1, 99]) if (img_sl > 0).any() else (0, 1))
    n = np.clip((img_sl - lo) / (hi - lo + 1e-8), 0, 1)
    rgb = np.stack([n, n, n], -1)
    colors = {2: [0.2, 0.4, 1.0], 1: [1.0, 0.9, 0.0], 0: [1.0, 0.1, 0.1]}
    for idx in [2, 1, 0]:
        m = seg_sl[idx] > 0
        if m.any():
            for c in range(3):
                rgb[..., c] = np.where(m, rgb[..., c]*(1-alpha) + colors[idx][c]*alpha, rgb[..., c])
    return np.clip(rgb, 0, 1)

def make_fig(r):
    t1c = r["img"][1]
    d, h, w = find_tumor_center(r["seg_gt"][2])
    si = [t1c[d], t1c[:, h], t1c[:, :, w]]
    sg = [r["seg_gt"][:, d], r["seg_gt"][:, :, h], r["seg_gt"][:, :, :, w]]
    sp = [r["seg_pred"][:, d], r["seg_pred"][:, :, h], r["seg_pred"][:, :, :, w]]
    cols = [f"Sagittal ({d})", f"Coronal ({h})", f"Axial ({w})"]
    rows = [("T1c MRI", None, None),
            ("GROUND TRUTH", sg, "#006400"),
            ("MODEL PREDICTION", sp, "#8B0000")]
    fig, axes = plt.subplots(3, 3, figsize=(18, 16))
    fig.suptitle(f"{r['case_id']} ({r['cohort']})  |  "
                 f"Dice ET={r['dice']['ET']:.3f}  TC={r['dice']['TC']:.3f}  "
                 f"WT={r['dice']['WT']:.3f}", fontsize=15, fontweight="bold", y=0.98)
    for col in range(3):
        for row in range(3):
            ax = axes[row, col]
            if row == 0:
                lo2, hi2 = (np.percentile(si[col][si[col]>0], [1,99])
                            if (si[col]>0).any() else (0,1))
                ax.imshow(si[col].T, cmap="gray", origin="lower", vmin=lo2, vmax=hi2)
            else:
                ax.imshow(overlay(si[col], rows[row][1][col]).transpose(1,0,2), origin="lower")
            ax.set_xticks([]); ax.set_yticks([])
            if row == 0: ax.set_title(cols[col], fontsize=14, fontweight="bold", pad=8)
            if col == 0:
                lbl, _, clr = rows[row]
                ax.text(-0.02, 0.5, lbl, transform=ax.transAxes, fontsize=13,
                        fontweight="bold", color=clr or "white", va="center",
                        ha="right", rotation=90,
                        bbox=dict(boxstyle="round,pad=0.3", facecolor="black",
                                  alpha=0.7) if row > 0 else None)
    from matplotlib.patches import Patch
    fig.legend(handles=[Patch(fc="#CC1A1A", label="ET"),
                        Patch(fc="#E6E600", ec="gray", label="TC"),
                        Patch(fc="#3366FF", label="WT")],
               loc="lower center", ncol=3, fontsize=12,
               bbox_to_anchor=(0.5, 0.005), frameon=True)
    plt.subplots_adjust(left=0.08, right=0.98, top=0.93, bottom=0.05,
                        hspace=0.08, wspace=0.05)
    return fig

# ================================================================== #
# Log everything to the training MLflow run                           #
# ================================================================== #
with mlflow.start_run(run_id=run_id):

    # --- Per-case dice metrics ---
    for r in results:
        for region, score in r["dice"].items():
            mlflow.log_metric(f"eval/{r['cohort']}/{region}", score)

    # --- Per-cohort averages ---
    for cohort in COHORTS:
        cr = [r for r in results if r["cohort"] == cohort]
        if cr:
            for reg in REGIONS:
                mlflow.log_metric(f"eval/{cohort}/avg_{reg}",
                                  np.mean([r["dice"][reg] for r in cr]))
            mlflow.log_metric(f"eval/{cohort}/avg_all",
                              np.mean([r["dice"][rg] for r in cr for rg in REGIONS]))

    # --- Global averages ---
    for reg in REGIONS:
        mlflow.log_metric(f"eval/mean_{reg}",
                          np.mean([r["dice"][reg] for r in results]))
    overall = np.mean([r["dice"][rg] for r in results for rg in REGIONS])
    mlflow.log_metric("eval/mean_dice", overall)
    print(f"\nOverall mean dice: {overall:.3f}")

    # --- Visualization PNGs ---
    for r in results:
        fig = make_fig(r)
        artifact_name = f"eval_visualizations/{r['cohort']}_{r['case_id']}.png"
        mlflow.log_figure(fig, artifact_name)
        plt.close(fig)
        print(f"  Logged {artifact_name}")

print("\nAll eval metrics and visualizations logged to MLflow.")